In [ ]:
# 1. INSTALL & IMPORT LIBRARY
!pip install PySastrawi -q

import pandas as pd
import numpy as np
import math
import re
from collections import defaultdict
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory


In [ ]:
# 2. LOAD DATA

from google.colab import files
uploaded = files.upload()    # Upload file Tugas_TKI__1_.xlsx

df = pd.read_excel('Tugas_TKI_1.xlsx', header=1)
df.columns = ['No', 'Dokumen']
df = df.dropna(subset=['Dokumen'])
df['No'] = df['No'].astype(int)

documents_raw = df['Dokumen'].tolist()

print(f"Jumlah dokumen  : {len(documents_raw)}")
print(f"Contoh dokumen 1: {documents_raw[0][:80]}...")

Saving Tugas_TKI_1.xlsx to Tugas_TKI_1 (1).xlsx
Jumlah dokumen  : 50
Contoh dokumen 1: anak ku sejak suntik polio gampang sakit. baru seminggu yg lalu batuk pilek, kel...


In [ ]:
# 3. PREPROCESSING — CASE FOLDING, HAPUS TANDA BACA, STOP-WORD REMOVAL

STOPWORDS = {
    'yang','dan','di','ke','dari','ini','itu','dengan','untuk','adalah','ada',
    'juga','tidak','sudah','saya','aku','kamu','dia','kami','kita','mereka',
    'pada','dalam','ya','aja','nya','nih','tuh','gak','ga','gue','gw','lo',
    'lah','deh','sih','kok','dong','kan','si','tapi','kalau','kalo','karena',
    'krn','jadi','jd','lebih','atau','bisa','buat','sama','mau','baru','udah',
    'sdh','apa','gimana','banget','bgt','banyak','semua','saja','pun','lagi',
    'terus','trus','memang','emang','nggak','enggak','belum','blm','karna',
    'dgn','utk','yg','dr','dlm','pd','tp','kl','sy','org','dg','dpt','jgn',
    'tsb','bs','klo','abis','habis','kayak','kayanya','kak','kk','ku','tau',
    'tahu','bilang','kata','orang','pake','pakai','terus','udah','mah','dah'
}

def preprocess(text):
    """Case folding → hapus tanda baca → tokenisasi → hapus stopwords."""
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens
              if t not in STOPWORDS and len(t) > 2]
    return tokens


print("Fungsi preprocessing berhasil didefinisikan.")
print(f"Contoh input : {documents_raw[0][:60]}...")
print(f"Contoh output: {preprocess(documents_raw[0])[:8]}")

Fungsi preprocessing berhasil didefinisikan.
Contoh input : anak ku sejak suntik polio gampang sakit. baru seminggu yg l...
Contoh output: ['anak', 'sejak', 'suntik', 'polio', 'gampang', 'sakit', 'seminggu', 'lalu']


In [ ]:
# 4. STEMMING — MENGGUNAKAN PYSASTRAWI (Bahasa Indonesia)

factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stem_tokens(tokens):
    """Mengubah setiap token ke bentuk kata dasar menggunakan Sastrawi."""
    return [stemmer.stem(t) for t in tokens]


corpus = [stem_tokens(preprocess(doc)) for doc in documents_raw]

print("Stemming selesai diterapkan ke seluruh corpus.")
print(f"\nContoh Dokumen 1:")
print(f"  Sebelum stem: {preprocess(documents_raw[0])[:7]}")
print(f"  Sesudah stem : {corpus[0][:7]}")

Stemming selesai diterapkan ke seluruh corpus.

Contoh Dokumen 1:
  Sebelum stem: ['anak', 'sejak', 'suntik', 'polio', 'gampang', 'sakit', 'seminggu']
  Sesudah stem : ['anak', 'sejak', 'suntik', 'polio', 'gampang', 'sakit', 'minggu']


In [ ]:
# 5. INVERTED INDEX

inverted_index = defaultdict(list)

for doc_id, tokens in enumerate(corpus):
    for term in set(tokens):              # set() agar satu term tidak dicatat
        inverted_index[term].append(doc_id + 1)   # dua kali di dokumen yang sama

# Urutkan posting list setiap term
for term in inverted_index:
    inverted_index[term] = sorted(inverted_index[term])


print(f"Inverted Index berhasil dibangun.")
print(f"Jumlah term unik: {len(inverted_index)}")
print(f"\nContoh Inverted Index:")
for term in list(inverted_index.keys())[:5]:
    print(f"  '{term}': {inverted_index[term]}")

Inverted Index berhasil dibangun.
Jumlah term unik: 654

Contoh Inverted Index:
  'polio': [1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 21, 22, 24, 25, 26, 27, 28, 31, 33, 34, 35, 36, 37, 38, 42, 43, 44, 45, 46, 47, 48, 49, 50]
  'lalu': [1, 5, 22, 33]
  'anak': [1, 5, 7, 12, 18, 20, 23, 27, 29, 30, 31, 32, 33, 38, 39, 41]
  'sejak': [1, 38]
  'hari': [1, 5, 15, 25, 50]


In [ ]:
# 6. FUNCTION TF (Log Frequency Weighting)
# Rumus: w = 1 + log10(tf) jika tf > 0, else 0

def tf(word, document):
    """Log frequency weighting: 1 + log10(tf) jika tf > 0, else 0."""
    count = document.count(word)
    if count > 0:
        return 1 + math.log10(count)
    return 0


print("Fungsi TF berhasil didefinisikan.")
print(f"Contoh: tf('polio', corpus[0]) = {tf('polio', corpus[0]):.4f}")
print(f"Contoh: tf('batuk', corpus[0]) = {tf('batuk', corpus[0]):.4f}")

Fungsi TF berhasil didefinisikan.
Contoh: tf('polio', corpus[0]) = 1.0000
Contoh: tf('batuk', corpus[0]) = 1.3010


In [ ]:
# 7. fungsi IDF (Inverse Document Frequency — Smooth)
# Rumus: idf = log10((N+1) / (df+1)) + 1

def idf(word, corpus):
    """IDF dengan Laplace smoothing agar aman dari division by zero."""
    N        = len(corpus) + 1                                      # total dok + 1
    df_count = sum(1 for doc in corpus if word in doc) + 1          # dok dengan kata + 1
    return math.log10(N / df_count) + 1


print("Fungsi IDF berhasil didefinisikan.")
print(f"Contoh: idf('polio', corpus)  = {idf('polio', corpus):.4f}")
print(f"Contoh: idf('batuk', corpus) = {idf('batuk', corpus):.4f}")

Fungsi IDF berhasil didefinisikan.
Contoh: idf('polio', corpus)  = 1.0948
Contoh: idf('batuk', corpus) = 2.2304


In [ ]:
# 8. FUNCTION TF-IDF

def tf_idf(word, document, corpus):
    """Bobot akhir: TF x IDF."""
    return tf(word, document) * idf(word, corpus)


print("Fungsi TF-IDF berhasil didefinisikan.")
print(f"Contoh: tf_idf('polio', corpus[0], corpus)  = {tf_idf('polio', corpus[0], corpus):.4f}")
print(f"Contoh: tf_idf('batuk', corpus[0], corpus) = {tf_idf('batuk', corpus[0], corpus):.4f}")

Fungsi TF-IDF berhasil didefinisikan.
Contoh: tf_idf('polio', corpus[0], corpus)  = 1.0948
Contoh: tf_idf('batuk', corpus[0], corpus) = 2.9019


In [ ]:
# 9. FUNCTION VSM — COSINE SIMILARITY
# Rumus: cos(q,d) = (q . d) / (|q| x |d|)

def cosine_similarity(query, document, corpus):
    """Hitung cosine similarity antara query dan dokumen."""

    words = set(query)

    dot_product = sum(
        tf_idf(w, query, corpus) * tf_idf(w, document, corpus)
        for w in words
    )

    mag_query = math.sqrt(
        sum(tf_idf(w, query, corpus) ** 2 for w in words)
    )

    mag_doc = math.sqrt(
        sum(tf_idf(w, document, corpus) ** 2 for w in set(document))
    )

    # Hindari pembagian dengan nol
    if mag_query == 0 or mag_doc == 0:
        return 0.0

    return dot_product / (mag_query * mag_doc)



In [ ]:
# 10. INPUT QUERY

query_input = input("Masukkan query: ")

# Query diproses sama seperti dokumen: preprocessing + stemming
query = stem_tokens(preprocess(query_input))

print(f"\nQuery asli              : {query_input}")
print(f"Query setelah preprocess: {query}")

Masukkan query: indonesia umumkan klb polio berakhir, tapi klb campak belum berakhir dan sebagiannya meninggal dunia

Query asli              : indonesia umumkan klb polio berakhir, tapi klb campak belum berakhir dan sebagiannya meninggal dunia
Query setelah preprocess: ['indonesia', 'umum', 'klb', 'polio', 'akhir', 'klb', 'campak', 'akhir', 'bagi', 'tinggal', 'dunia']


In [ ]:
# 11. CARI DOKUMEN VIA INVERTED INDEX + HITUNG COSINE SIMILARITY

# Gunakan inverted index untuk menemukan kandidat dokumen
candidate_ids = set()
for word in query:
    if word in inverted_index:
        candidate_ids.update(inverted_index[word])

print(f"Dokumen kandidat (via Inverted Index): {sorted(candidate_ids)}")
print(f"Jumlah kandidat : {len(candidate_ids)} dokumen\n")

# Hitung cosine similarity hanya untuk dokumen kandidat
results = []
for doc_id in candidate_ids:
    doc = corpus[doc_id - 1]
    score = cosine_similarity(query, doc, corpus)
    score = round(score, 5)
    results.append((doc_id, documents_raw[doc_id - 1], score))

# Sort descending
results = sorted(results, key=lambda x: x[2], reverse=True)

print(f"Skor berhasil dihitung untuk {len(results)} dokumen.")

Dokumen kandidat (via Inverted Index): [1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 21, 22, 24, 25, 26, 27, 28, 29, 31, 33, 34, 35, 36, 37, 38, 42, 43, 44, 45, 46, 47, 48, 49, 50]
Jumlah kandidat : 41 dokumen

Skor berhasil dihitung untuk 41 dokumen.


In [ ]:
# 12. GROUPING DAN TAMPILKAN RANKING

grouped = {}
for idx, doc, score in results:
    grouped.setdefault(score, []).append((idx, doc))

print(f"\n{'='*65}")
print(f"  HASIL PENCARIAN untuk query: '{query_input}'")
print(f"  Query (stemmed)            : {query}")
print(f"{'='*65}")

rank  = 1
shown = 0

for score, docs in grouped.items():
    if score == 0:
        continue

    doc_ids = ', '.join([f"D{d[0]}" for d in docs])
    print(f"\nRank {rank} | Dokumen: {doc_ids} | Skor: {score}")
    print("-" * 65)

    for d in docs:
        preview = d[1][:120] + "..." if len(d[1]) > 120 else d[1]
        print(f"  D{d[0]}: {preview}")

    rank  += 1
    shown += 1

if shown == 0:
    print("\nTidak ada dokumen yang relevan dengan query tersebut.")



  HASIL PENCARIAN untuk query: 'indonesia umumkan klb polio berakhir, tapi klb campak belum berakhir dan sebagiannya meninggal dunia'
  Query (stemmed)            : ['indonesia', 'umum', 'klb', 'polio', 'akhir', 'klb', 'campak', 'akhir', 'bagi', 'tinggal', 'dunia']

Rank 1 | Dokumen: D6 | Skor: 0.26101
-----------------------------------------------------------------
  D6: iyhh adik aku dlu dipolio ,udh nya lumpuh ,sampai meninggal dunia😢

Rank 2 | Dokumen: D46 | Skor: 0.15489
-----------------------------------------------------------------
  D46: Vaksin BCG jadi kena TBC…
Vaksin polio, jadi KLB polio (yg 100% dari strain 💉 - gak ada dari wild polio)…
Solusi medis: ...

Rank 3 | Dokumen: D37 | Skor: 0.14322
-----------------------------------------------------------------
  D37: Mungkin rakyat Malaysia belum terjangkit virus tiktok.

Indonesia sudah pernah bebas polio, artinya dulu orang2 taat per...

Rank 4 | Dokumen: D8 | Skor: 0.13453
----------------------------------------------